# Building Resilient Agents with Kalibr

**Partner cookbook — [Kalibr Systems](https://kalibr.systems)**

---

## The Problem with Single-Model Agents

Most OpenAI-powered agents are written like this:

```python
from openai import OpenAI
client = OpenAI()
response = client.chat.completions.create(model="gpt-4o", messages=[...])
```

This works in demos. In production:
- Rate limits fire at unpredictable times
- Provider latency spikes during peak load
- Partial outages affect specific model versions
- Your agent silently degrades or errors out — often with HTTP 200 responses

**What you actually need:** route away from degraded paths automatically, and route back when they recover.

---

## What is Kalibr?

Kalibr captures the outcome of every LLM call (success, failure, quality score 0–1), learns which model+parameter combinations work best for each goal, and routes the next call to the optimal path — without redeployment.

**Not:** observability (Langfuse/LangSmith), model gateway (LiteLLM/OpenRouter), or a proxy.

**Classification:** execution path routing based on outcome signals.

In [ ]:
!pip install kalibr openai anthropic -q

## Setup

Set your API keys in the environment before running this notebook.

Required environment variables:
- `OPENAI_API_KEY` — [OpenAI platform](https://platform.openai.com/api-keys)
- `ANTHROPIC_API_KEY` — [Anthropic console](https://console.anthropic.com/) (fallback model)
- `KALIBR_API_KEY` — [Kalibr dashboard](https://dashboard.kalibr.systems) or run `kalibr auth`
- `KALIBR_TENANT_ID` — [Kalibr dashboard](https://dashboard.kalibr.systems)

In [ ]:
import os

# Verify required environment variables are set
required_vars = ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "KALIBR_API_KEY", "KALIBR_TENANT_ID"]
missing = [v for v in required_vars if not os.environ.get(v)]
if missing:
    raise EnvironmentError(
        f"Missing required environment variables: {missing}\n"
        "Set them before running this notebook. See README for details."
    )
print("\u2713 All required environment variables are set")

---

## Important: Import Order

`import kalibr` must come **before** any OpenAI or Anthropic import. It instruments SDK clients at import time via monkey-patching. The cell below sets this up for the entire notebook — both the 'before' and 'after' demos.

In [ ]:
# kalibr MUST be first — instruments OpenAI and Anthropic SDK clients at import time
import kalibr  # noqa: F401

from openai import OpenAI
from kalibr import Router

print("\u2713 Kalibr instrumentation active")

---

## Part 1: The Fragile Pattern

This is the standard pattern most agents use. It works — until the model degrades.

```python
# ❌ Fragile — hardcoded to one model, no fallback
client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": query}],
    max_tokens=300,
)
```

When GPT-4o has a degradation event (rate limits, latency spikes, partial outage), this agent fails. Recovery requires human intervention: notice the problem, update the code, redeploy.

**Measured success rate during a 70% GPT-4o degradation event: 16–36%**

---

## Part 2: The Resilient Pattern (Kalibr)

Two changes from the fragile pattern:
1. `import kalibr` — already done above (must be first in the process)
2. `Router` instead of bare `OpenAI` client

The response interface is **identical**: `response.choices[0].message.content`

In [ ]:
support_router = Router(
    goal="customer_support_response",
    paths=["gpt-4o", "claude-sonnet-4-20250514"],
    success_when=lambda output: output is not None and len(output) > 30,
)

def support_agent_resilient(user_query: str) -> str:
    """
    Customer support agent with Kalibr routing.

    Kalibr selects the model with the highest success rate for this goal.
    If GPT-4o is degraded, it routes to Claude Sonnet automatically.
    """
    response = support_router.completion(
        messages=[
            {"role": "system", "content": "You are a helpful customer support agent."},
            {"role": "user", "content": user_query}
        ],
        max_tokens=300,
    )
    return response.choices[0].message.content

result = support_agent_resilient("How do I reset my password?")
print(result)

---

## Part 3: Outcome Reporting

For tasks where structural validation isn't enough, report outcomes explicitly.
This gives Kalibr a richer signal — quality score 0–1, not just pass/fail.

In [ ]:
quality_router = Router(
    goal="customer_support_quality",
    paths=["gpt-4o", "claude-sonnet-4-20250514", "gpt-4o-mini"],
    # No success_when — we report manually for finer control
)

response = quality_router.completion(
    messages=[
        {"role": "system", "content": "You are a helpful customer support agent."},
        {"role": "user", "content": "How do I update my billing information?"}
    ],
    max_tokens=300,
)
output = response.choices[0].message.content

# Your domain-specific quality check
has_steps = any(w in output.lower() for w in ["click", "go to", "select", "navigate"])
appropriate_length = 50 < len(output) < 600
quality_score = (int(has_steps) + int(appropriate_length)) / 2.0

quality_router.report(
    success=quality_score >= 0.5,
    score=quality_score,
    failure_category=None if quality_score >= 0.5 else "user_unsatisfied",
)
print(f"Quality score: {quality_score:.2f}")
print(f"Response: {output}")

---

## Benchmark Results

From Kalibr's resilience benchmark during simulated GPT-4o degradation (70% failure rate injected at task 16 of 50):

| Agent Setup | Task Success Rate During Degradation |
|---|---|
| Hardcoded `gpt-4o` | 16–36% |
| Retry with exponential backoff | 40–55% |
| Kalibr-routed (`gpt-4o` + `claude-sonnet-4-20250514`) | 88–100% |

Full benchmark methodology: [kalibr.systems/docs/benchmark](https://kalibr.systems/docs/benchmark)

---

## What Changed

| | Fragile Pattern | Kalibr Pattern |
|---|---|---|
| First import | `from openai import OpenAI` | `import kalibr` |
| Client | `OpenAI()` | `Router(goal, paths, success_when)` |
| Call | `client.chat.completions.create(model=..., ...)` | `router.completion(messages=...)` |
| Response | `response.choices[0].message.content` | identical |
| Fallback | None | Automatic |
| Adapts over time | No | Yes — learns from every call |

---

## Links

- [Kalibr docs](https://kalibr.systems/docs)
- [Resilience benchmark](https://kalibr.systems/docs/benchmark)
- [GitHub](https://github.com/kalibr-ai/kalibr-sdk-python)
- [Dashboard](https://dashboard.kalibr.systems)
- [PyPI](https://pypi.org/project/kalibr/)